# QM9 inference-featurizer cleanup — source-verified

Run in the **buhito** kernel, without training. This version is based on the supplied `transformers.py`.

Ordinary `GraphletTransformer.transform()` with `_in_fit_transform=False` regenerates graphlet locations and unseen-feature diagnostics. It does not read the saved `bi_fit_` training locations. Those locations can therefore be cleared for this inference workflow, alongside transient transform outputs and the fit/transform handoff cache.

The nested BFS featurizer, fitted vocabulary, feature indices/order, feature sizes, fitted flag, and representation configuration are preserved. Both original pickle files remain unchanged. A separate inference pickle is published only after round-trip checks of fitted metadata, sparse features, and graphlet locations pass.

This does not preserve access to historical training locations via `bi_fit_`: keep the original for training-history inspection. No source code is patched. File size and load-time improvements must be measured locally.


## 1. Imports and models directory

If you run this beside your original interpretability notebook, the default directory is `../models`. Edit `MODELS_DIR` if your saved featurizer is elsewhere. An existing `QM9_MODELS_DIR` environment variable takes precedence by default.


In [1]:
import gc
import os
import tempfile
from pathlib import Path
from time import perf_counter
from statistics import median

import joblib
import numpy as np
from buhito.converters import smiles_to_nx

MODELS_DIR = Path(
    os.environ.get("QM9_MODELS_DIR", str(Path.cwd().parent / "models"))
).expanduser().resolve()
# Example override:
# MODELS_DIR = Path(r"C:\Users\statl\Documents\GitHub\buhito-interpretability\models")

# Prefer the previously validated smaller copy to avoid loading the largest file again.
previous_clean = MODELS_DIR / "buhito_featurizer_clean.pkl"
ORIGINAL_PATH = previous_clean if previous_clean.is_file() else MODELS_DIR / "buhito_featurizer.pkl"
CLEAN_PATH = MODELS_DIR / "buhito_featurizer_inference.pkl"
OVERWRITE_PREVIOUS_CLEAN = False  # Set True to replace an earlier cleaned copy.
TEST_SMILES = ["FC(F)(F)F", "O=C1NN=CN=C1", "COC12CC1OC2CO"]

if not ORIGINAL_PATH.is_file():
    raise FileNotFoundError(f"Edit MODELS_DIR: no featurizer found at {ORIGINAL_PATH}")
if ORIGINAL_PATH.resolve() == CLEAN_PATH.resolve():
    raise ValueError("The cleaned file must have a different path from the original.")
if CLEAN_PATH.exists() and not OVERWRITE_PREVIOUS_CLEAN:
    raise FileExistsError(
        f"{CLEAN_PATH} already exists. Set OVERWRITE_PREVIOUS_CLEAN=True or choose a new name."
    )
print("Input (unchanged):", ORIGINAL_PATH)
print("Cleaned copy:", CLEAN_PATH)


Input (unchanged): C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer_clean.pkl
Cleaned copy: C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer_inference.pkl


## 2. Load the input and inspect retained state

The input defaults to your existing `buhito_featurizer_clean.pkl`, falling back to `buhito_featurizer.pkl`. This still requires one input load. It does not require rerunning the first cleanup notebook.

No attributes are removed in this cell. The later cleanup targets `bi_fit_`, `bi_transform_`, `_cache`, `_in_fit_transform`, `X_unseen_`, `n_unseen_`, `bit_ids_unseen_`, `bit_indices_unseen_`, `bit_sizes_unseen_`, and `transform_time_`.


In [2]:
load_started = perf_counter()
featurizer = joblib.load(ORIGINAL_PATH)
original_load_seconds = perf_counter() - load_started
original_size = ORIGINAL_PATH.stat().st_size

print(f"Input file: {original_size / 1024**2:.2f} MiB")
print(f"Initial load: {original_load_seconds:.2f} seconds")
if not hasattr(featurizer, "bi_transform_"):
    raise AttributeError("No bi_transform_ attribute found. Inspect this Buhito version before cleaning.")
print("Saved location-cache type:", type(featurizer.bi_transform_).__name__)
try:
    print("Saved location-cache entries:", len(featurizer.bi_transform_))
except TypeError:
    print("Saved location cache has no length.")

for name, value in vars(featurizer).items():
    try:
        length = len(value)
    except TypeError:
        length = None
    print(f"{name}: type={type(value).__name__}, shape={getattr(value, 'shape', None)}, length={length}")

# Single-molecule inference settings used in the source notebook.
featurizer.n_jobs = 1
featurizer.verbose = False
if not bool(featurizer.is_fitted_):
    raise ValueError("Expected a fitted transformer.")
if featurizer._in_fit_transform:
    raise ValueError("The saved transformer is still marked as being inside fit_transform.")
vocabulary_hash = joblib.hash((
    featurizer.bit_ids_, featurizer.bit_sizes_, featurizer.bit_indices_, featurizer.n_bits_
))


Input file: 530.13 MiB
Initial load: 297.13 seconds
Saved location-cache type: list
Saved location-cache entries: 0
featurizer: type=BFSGraphletFeaturizer, shape=None, length=None
return_dense: type=bool, shape=None, length=None
return_float: type=bool, shape=None, length=None
n_jobs: type=int, shape=None, length=None
chunk_size: type=str, shape=None, length=4
verbose: type=bool, shape=None, length=None
_cache: type=dict, shape=None, length=0
_in_fit_transform: type=bool, shape=None, length=None
n_bits_: type=int, shape=None, length=None
bit_ids_: type=list, shape=None, length=5370
bit_sizes_: type=ndarray, shape=(5370,), length=5370
bit_indices_: type=dict, shape=None, length=5370
n_unseen_: type=int, shape=None, length=None
X_unseen_: type=NoneType, shape=None, length=None
bit_ids_unseen_: type=list, shape=None, length=0
bit_indices_unseen_: type=dict, shape=None, length=0
bit_sizes_unseen_: type=list, shape=None, length=0
bi_fit_: type=tuple, shape=None, length=104664
is_fitted_: ty

## 3. Record reference outputs

Each SMILES is transformed separately, using explicit hydrogens as in model training. Graphlet occurrences are sorted for comparison while retaining duplicate instances. We compare all reported locations, not just nonzero feature counts.


In [3]:
def make_graph(smiles):
    converted = smiles_to_nx(smiles, add_hs=True, output_2d_pos=False)
    return converted[0] if isinstance(converted, (tuple, list)) else converted


def location_signature(bitinfo):
    return tuple(sorted(
        (
            repr(bit_id),
            tuple(sorted(
                tuple(sorted(int(atom) for atom in occurrence))
                for occurrence in occurrences
            )),
        )
        for bit_id, occurrences in bitinfo.items()
    ))


def snapshot(transformer, smiles):
    features = transformer.transform([make_graph(smiles)]).tocsr()
    if not hasattr(transformer, "bi_transform_") or len(transformer.bi_transform_) != 1:
        raise RuntimeError("Expected one graphlet-location dictionary after a one-molecule transform.")
    return features.copy(), location_signature(transformer.bi_transform_[0])


references = {smiles: snapshot(featurizer, smiles) for smiles in TEST_SMILES}
for smiles, (features, _) in references.items():
    print(f"Reference ready: {smiles}; shape={features.shape}; nonzero features={features.nnz}")


Reference ready: FC(F)(F)F; shape=(1, 5370); nonzero features=5
Reference ready: O=C1NN=CN=C1; shape=(1, 5370); nonzero features=68
Reference ready: COC12CC1OC2CO; shape=(1, 5370); nonzero features=83


## 4. Clear source-verified transient state, then round-trip validate

The source never reads `bi_fit_` in ordinary `transform()`. The transform regenerates `bi_transform_` and the unseen-feature outputs. `_cache` is used only for the fit/transform handoff when `_in_fit_transform=True`; we retain inference mode with that flag set to False.

We empty the saved location tuples, reset unseen outputs, and clear the handoff cache. Fitted feature metadata and the nested BFS featurizer are preserved. The temporary pickle is validated before publication; any preexisting output remains unchanged on failure. The saved pickle stays clean even though validation transforms regenerate temporary state in memory.


In [4]:
featurizer.bi_fit_ = ()
featurizer.bi_transform_ = ()
featurizer._cache = {}
featurizer._in_fit_transform = False
featurizer.n_unseen_ = 0
featurizer.X_unseen_ = None
featurizer.bit_ids_unseen_ = []
featurizer.bit_indices_unseen_ = {}
featurizer.bit_sizes_unseen_ = np.array([], dtype=int)
featurizer.transform_time_ = 0.0
handle, temporary_name = tempfile.mkstemp(
    prefix="featurizer_cleanup_", suffix=".pkl", dir=MODELS_DIR
)
os.close(handle)
temporary_path = Path(temporary_name)
clean = None

try:
    joblib.dump(featurizer, temporary_path, compress=0)
    del featurizer
    gc.collect()

    load_started = perf_counter()
    clean = joblib.load(temporary_path)
    clean_load_seconds = perf_counter() - load_started

    if len(clean.bi_transform_) != 0:
        raise AssertionError("The saved cleaned copy still contains graphlet-location entries.")
    if len(clean.bi_fit_) != 0 or clean._cache or clean._in_fit_transform:
        raise AssertionError("Training locations or fit/transform cache were not cleared.")
    if clean.X_unseen_ is not None or clean.n_unseen_ != 0:
        raise AssertionError("Unseen-feature outputs were not cleared.")
    if joblib.hash((clean.bit_ids_, clean.bit_sizes_, clean.bit_indices_, clean.n_bits_)) != vocabulary_hash:
        raise AssertionError("The fitted vocabulary or feature ordering changed.")

    for smiles, (expected_features, expected_locations) in references.items():
        actual_features, actual_locations = snapshot(clean, smiles)
        if actual_features.shape != expected_features.shape:
            raise AssertionError(f"Feature shape changed: {smiles}")
        if (actual_features != expected_features).nnz != 0:
            raise AssertionError(f"Feature values changed: {smiles}")
        if actual_locations != expected_locations:
            raise AssertionError(f"Graphlet locations changed: {smiles}")
        print("Passed:", smiles)

    # Recheck the destination in case it was created during validation.
    if CLEAN_PATH.exists() and not OVERWRITE_PREVIOUS_CLEAN:
        raise FileExistsError(f"Refusing to replace existing cleaned copy: {CLEAN_PATH}")
    os.replace(temporary_path, CLEAN_PATH)
    clean_size = CLEAN_PATH.stat().st_size
    print("Validated cleaned copy saved:", CLEAN_PATH)
    print(f"Original: {original_size / 1024**2:.2f} MiB")
    print(f"Cleaned:  {clean_size / 1024**2:.2f} MiB")
    print(f"Size reduction: {100 * (1 - clean_size / original_size):.1f}%")
    print(f"Initial loads: original={original_load_seconds:.2f}s; cleaned={clean_load_seconds:.2f}s")
    print("These initial load times are not a controlled benchmark; OS disk caching can affect them.")
finally:
    temporary_path.unlink(missing_ok=True)
    # Validation transforms populate the in-memory cache again. Release that object.
    clean = None
    gc.collect()


Passed: FC(F)(F)F
Passed: O=C1NN=CN=C1
Passed: COC12CC1OC2CO
Validated cleaned copy saved: C:\Users\statl\Documents\GitHub\buhito-interpretability\models\buhito_featurizer_inference.pkl
Original: 530.13 MiB
Cleaned:  0.22 MiB
Size reduction: 100.0%
Initial loads: original=297.13s; cleaned=0.06s
These initial load times are not a controlled benchmark; OS disk caching can affect them.


## 5. Optional repeated loading comparison

Set `RUN_BENCHMARK=True` to alternate loading the two files three times. Only one loaded object is retained at a time. This can take a while. Median times are reported, but the test primarily reflects warm filesystem-cache performance; do not interpret it as an unbiased cold-start benchmark.


In [5]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    times = {"original": [], "cleaned": []}
    for repetition in range(3):
        order = [("original", ORIGINAL_PATH), ("cleaned", CLEAN_PATH)]
        if repetition % 2:
            order.reverse()
        for label, path in order:
            gc.collect()
            started = perf_counter()
            loaded_object = joblib.load(path)
            seconds = perf_counter() - started
            times[label].append(seconds)
            del loaded_object
            print(f"Run {repetition + 1}, {label}: {seconds:.2f}s")
    for label, values in times.items():
        print(f"Median {label} load: {median(values):.2f}s")
else:
    print("Benchmark skipped. Set RUN_BENCHMARK=True to run it.")


Benchmark skipped. Set RUN_BENCHMARK=True to run it.


## 6. Use the validated inference copy

1. Stop Shiny and keep backups of your existing files.
2. Only after all checks pass, copy `buhito_featurizer_inference.pkl` to the app's selected models directory as `buhito_featurizer.pkl`.
3. If you use separate HOMO/LUMO featurizers, clean each separately and keep it paired with the model trained using that vocabulary.
4. Restart the server to clear old in-memory objects.
5. Compare a familiar molecule's prediction, signed contributions, raw score, and adjusted score. Disable Fast analysis and hold all search settings and seed fixed for partition comparisons.

Exact vocabulary and feature-vector equality preserves inputs to the unchanged models. This notebook checks sample feature vectors and graphlet locations; it does not run the actual trained models or partition searches. Historical training graphlet locations are intentionally absent in the inference copy.

If this file remains large, inspect the nested BFS featurizer next; it is retained unchanged here. No below-30-second loading guarantee is made. The source-verified cleanup was tested using the supplied GraphletTransformer with a deterministic fixture featurizer; your real pickle and chemistry environment still require validation here.
